### **Random Sampling in Hyperspecteral Image classification with Supervised Band Selection**
This Demo is an implementation of the iBEARDS and PEMROCKET methods for fast and accurate Hyperspectreal Image classification  

Please cite this work in the following manner:

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')
import sys
sys.path.insert(1, "/content/gdrive/My Drive/Colab Notebooks/HSI/Classification-of-Hyperspectral-Image-master/")
sys.path.insert(2, "/content/gdrive/My Drive/Colab Notebooks/rocket/.preprint/")
%cd /content/gdrive/My Drive/Colab Notebooks/HSI/AMS-M2ESL-main/

Mounted at /content/gdrive
/content/gdrive/My Drive/Colab Notebooks/HSI/AMS-M2ESL-main


USE this Fuction for SLOW S-NER

In [ ]:
from sklearn.linear_model import RidgeClassifierCV, LinearRegression, Lars, lars_path ,Ridge ,orthogonal_mp, OrthogonalMatchingPursuit
from scipy.special import comb, gammaln,  gammaincinv
def NER_sort(x,y,k_max,n,c,c1):
  A_S=[-2]
  # A_S_aided=[]
  y=y-np.mean(y)
  y=y/np.std(y)
  Y_NORM = (np.linalg.norm(y)**2)/n
  SNR_hat=[]
  R_emp=[Y_NORM]
  L=x.shape[1]
  path=lars_path(x,y,method='lar',max_iter=70)[1]
  k_max=int(np.min([len(path),k_max]))
  L=int(k_max)
  # N_A_S = list(np.arange(L))
  N_A_S = list(path)
  k=0
  Test=1>0
  iii=0
  A_S_S_NER=[-2]
  while ((k<k_max)&(Test==True)):
    R_emp_can=[]
    SNR_hat_can=[]
    for i in range(L-k):
      A_S_can=list(set(A_S)|set([N_A_S[i]]))
      A_S_can=list(set(A_S_can)-{-2})
      A_S_can=list(np.sort(np.array(A_S_can)))
      # print(A_S_temp)
      X_A=x[:,A_S_can]
      AAA = LinearRegression( fit_intercept = False)
      AAA.fit(X_A, y)
      r=np.matmul(X_A,np.transpose(AAA.coef_))-y
      SNR_hat_can.append(np.var(r))
      R_emp_can.append((np.linalg.norm(r)**2)/n)
    idx=np.argmin(np.array(R_emp_can))

    k=k+1
    A_S.append(N_A_S[idx])
    A_S=list(set(A_S)-{-2})

    R_emp.append(R_emp_can[idx])
    print("R_slow")
    print(np.array(R_emp))

    SNR_hat=SNR_hat_can[idx]

    BOUND =c*((1)/n)*np.array(SNR_hat)*gammaincinv(1/2,1-((1/n)*np.array(SNR_hat)))

    Test=R_emp[-2]-R_emp[-1]>BOUND
    if k<15:
      Test=1>0

    extra=N_A_S[idx]

    N_A_S=list(set(N_A_S)-{extra})
  A_S_S_NER=list(set(A_S)-{extra})

  A_S_S_NER_A=np.array(A_S_S_NER)
  print("S-NER Done")

  return list(A_S_S_NER_A)

In [2]:
def S_NER(train_data_index, X, y,ccc):
  newX = np.reshape(X, (-1, X.shape[2]))
  y1= np.reshape(y, -1)
  newXX=(newX-np.mean(newX))/np.std(newX)
  X_train_NER=newXX[train_data_index,:]
  y_train_NER=y1[train_data_index]
  y_train_NER=(y_train_NER-np.mean(y_train_NER))/np.std(y_train_NER)

  # SET_NER = apply_NER(X_train_NER,y_train_NER,L,ccc)

  # SET_NER = NER_sort(X_train_NER,y_train_NER,X.shape[2]/2,len(train_data_index),ccc,1)

  SET_NER = fast_NER_sort(X_train_NER,y_train_NER,X.shape[2]/2,len(train_data_index),ccc,1)

  return SET_NER

Transforming HSI patches to 1 dimensional vectors

In [3]:
def createPatches_concatenated(X, y, Sep_labels, windowSize=5, removeZeroLabels = True):
    margin = int((windowSize - 1) / 2)
    SHAPE=X.shape
    Patch_length=len(np.array(Sep_labels))
    X = padWithZeros(X, margin=margin)
    x_offset = margin
    y_offset = margin

    # split patches
    patchesData = np.zeros((Patch_length, windowSize*windowSize*SHAPE[2]))
    patchesLabels = np.zeros((Patch_length))
    patchIndex = 0
    i=0
    One_to_Tow_dim_cordinate=np.zeros([SHAPE[0]*SHAPE[1],2],dtype=int)
    for r in range(SHAPE[0]):
      for c in range(SHAPE[1]):
        One_to_Tow_dim_cordinate[i,:]=np.array([r,c])
        i=i+1


    for i in Sep_labels:
      r=One_to_Tow_dim_cordinate[i,0];c=One_to_Tow_dim_cordinate[i,1]
      patch = X[r :r + 2*margin + 1, c :c + 2*margin + 1]
      patch = np.reshape(patch,(patch.shape[0]*patch.shape[1]*patch.shape[2]))
      patchesData[patchIndex, :] = patch
      patchesLabels[patchIndex] = y[r, c]
      patchIndex=patchIndex+1


    if removeZeroLabels:
        patchesData = patchesData[patchesLabels>0,:,:,:]
        patchesLabels = patchesLabels[patchesLabels>0]
        patchesLabels -= 1
    return patchesData, patchesLabels


USE this Function for Fast S-NER

In [5]:
from sklearn.linear_model import RidgeClassifierCV, LinearRegression, Lars, lars_path
from scipy.special import gammaincinv
def fast_NER_sort(x,y,k_max,n,c,c1):
  A_S=[]
  y=y-np.mean(y)
  y=y/np.std(y)
  Y_NORM = (np.linalg.norm(y)**2)/n
  SNR_hat=[]
  R_emp=[Y_NORM]
  L=x.shape[1]
  path=lars_path(x,y,method='lar',max_iter=70)[1]
  k_max=int(np.min([len(path),k_max]))
  L=int(k_max)
  # N_A_S = list(np.arange(L))
  N_A_S = list(path)
  k=1
  Test=1>0
  iii=0
  A_S_S_NER=[]
  R_emp_can=[]
  SNR_hat_can=[]
  for i in range(L):
    X_A=x[:,list({N_A_S[i]})]
    AAA = LinearRegression( fit_intercept = False)
    AAA.fit(X_A, y)
    r=np.matmul(X_A,np.transpose(AAA.coef_))-y
    SNR_hat_can.append(np.var(r))
    R_emp_can.append((np.linalg.norm(r)**2)/n)
  idx=np.argmin(np.array(R_emp_can))
  A_S.append(N_A_S[idx])
  XX=x[:,N_A_S[idx]]
  N_A_S=list(set(N_A_S)-{N_A_S[idx]})
  R_emp.append(R_emp_can[idx])
  SNR_hat=SNR_hat_can[idx]
  A1=np.array([[1/np.dot(XX,XX)]])
  while ((k<k_max)&(Test==True)):
    R_emp_can=[]
    SNR_hat_can=[]
    AA_store=("empty",)
    for i in range(L-k):
      A_S_can=list(A_S)
      A_S_can.append(N_A_S[i])
      X_A=x[:,A_S_can]
      AA1=fast_invers(A1,X_A)
      AA_store=AA_store+(AA1,)
      LS_answer=np.matmul(np.matmul(AA1,np.transpose(X_A)),y)
      r=y-np.matmul(X_A,LS_answer)
      SNR_hat_can.append(np.var(r))
      R_emp_can.append((np.linalg.norm(r)**2)/n)
    idx=np.argmin(np.array(R_emp_can))
    A1=AA_store[idx+1]
    k=k+1
    A_S.append(N_A_S[idx])
    R_emp.append(R_emp_can[idx])
    SNR_hat=SNR_hat_can[idx]
    BOUND =c*((1)/n)*np.array(SNR_hat)*gammaincinv(1/2,1-((1/n)*np.array(SNR_hat)))
    Test=R_emp[-2]-R_emp[-1]>BOUND
    if k<15:
      Test=1>0
    extra=N_A_S[idx]
    N_A_S=list(set(N_A_S)-{extra})
  A_S_S_NER=list(set(A_S)-{extra})
  A_S_S_NER_A=np.array(A_S_S_NER)
  print("S-NER Done")
  return list(A_S_S_NER_A)

In [4]:
def fast_invers(A,XX):
  n,k=XX.shape
  index=list({k-1})
  X2 = XX[:,index]
  b = np.zeros([k-1,1])
  b =np.matmul(np.transpose(XX[:,:k-1]),X2)
  Ab= np.zeros([k-1,1])
  Ab=np.matmul(A,b)
  c = np.zeros([1,k-1])
  c  = np.matmul(np.transpose(X2),XX[:,:k-1])
  cA= np.zeros([1,k-1])
  cA=np.matmul(c,A)
  dcAb=1/(np.matmul(np.transpose(X2),X2)-np.matmul(cA,b))
  AA=np.zeros([k,k])
  AA[:k-1,:k-1]=A + np.matmul(Ab,cA)*dcAb
  AA[:k-1,k-1]=(-Ab*dcAb)[:,0]
  AA[k-1,:k-1]=-cA*dcAb
  AA[k-1,k-1]=dcAb
  return AA

In [6]:
!pip install sktime

import sktime
import sklearn

from sktime.transformations.panel.rocket import MiniRocket
from sklearn.preprocessing import StandardScaler
Rr_IP=[0.01,0.025,0.05,0.075,0.1,0.15,0.2]
Rr_IP=[0.01]

Rr_UP=[0.01,0.025,0.05,0.075,0.1,0.15,0.2]
Rr_UP=[0.075]
Rr_SL=[0.01,0.025,0.05,0.075,0.1,0.15,0.2]
Rr_SL=[0.025,75]
Rr_HU=[0.15,0.2]

order='spec1'
DR_list=[0]
DR_flag=DR_list[0]
# C_store=[0.01,0.02,0.05,0.2,0.4,0.1]
C_store=[0.01]
cc=C_store[0]
# Num_seg=185
# Method_store=[7]
Patch_Size_store=[10000]
D_fl=0
if D_fl==0:
  Rr_list=Rr_IP
if D_fl==1:
  Rr_list=Rr_UP
if D_fl==2:
  Rr_list=Rr_SL
if D_fl==3:
  Rr_list=Rr_HU
Rr=Rr_list[0]

Patch_Size=9
N_kernels=Patch_Size_store[0]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 11.5 MB/s eta 0:00:00


/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [12]:
import scipy.io as sio
import os
from sklearn import preprocessing
from operator import truediv
def load_data(data_set_name, data_path):
    if data_set_name == 'IP':
        data = sio.loadmat(os.path.join(data_path, 'IP', 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(data_path, 'IP', 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif data_set_name == 'UP':
        data = sio.loadmat(os.path.join(data_path, 'UP', 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(data_path, 'UP', 'PaviaU_gt.mat'))['paviaU_gt']
    elif data_set_name == 'SL':
        data = sio.loadmat(os.path.join(data_path, 'SL', 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(data_path, 'SL', 'Salinas_gt.mat'))['salinas_gt']

    return data, labels

def standardization(data):
    height, width, bands = data.shape
    data = np.reshape(data, [height * width, bands])
    # data=preprocessing.scale(data) #
    # data = preprocessing.MinMaxScaler().fit_transform(data)
    data = preprocessing.StandardScaler().fit_transform(data)  #

    data = np.reshape(data, [height, width, bands])
    return data


def sampling(ratio_list, num_list, gt_reshape, class_count, Flag):
    all_label_index_dict, train_label_index_dict, test_label_index_dict = {}, {}, {}
    all_label_index_list, train_label_index_list, test_label_index_list = [], [], [],

    for cls in range(class_count):  # [0-15]
        cls_index = np.where(gt_reshape == cls + 1)[0]
        all_label_index_dict[cls] = list(cls_index)

        np.random.shuffle(cls_index)

        if Flag == 0:  # Fixed proportion for each category
            train_index_flag = max(int(ratio_list[0] * len(cls_index)), 3)  # at least 3 samples per class]
        # Split by num per class
        elif Flag == 1:  # Fixed quantity per category
            if len(cls_index) > num_list[0]:
                train_index_flag = num_list[0]
            else:
                train_index_flag = 15

        train_label_index_dict[cls] = list(cls_index[:train_index_flag])
        test_label_index_dict[cls] = list(cls_index[train_index_flag:])

        train_label_index_list += train_label_index_dict[cls]
        test_label_index_list += test_label_index_dict[cls]
        all_label_index_list += all_label_index_dict[cls]

    return train_label_index_list, test_label_index_list, all_label_index_list

def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX

def AA_ECA(confusion_matrix):
    # get diagonal element
    diag_list = np.diag(confusion_matrix)
    row_sum_list = np.sum(confusion_matrix, axis=1)
    each_per_acc = np.nan_to_num(truediv(diag_list, row_sum_list))
    avg_acc = np.mean(each_per_acc)

    return each_per_acc, avg_acc

In [13]:
# # iBEARDS and PEM-ROCKET


import os
import time
import torch
import random
import numpy as np
from sklearn import metrics

# random seed setting
seed = 20

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("IFFF DRs in NER and AMS using GPU:")
print(device)

data_set_name_list = ['IP', 'UP', 'SL']
data_set_name = data_set_name_list[D_fl]

data_set_path = os.path.join(os.getcwd(), 'data')

# seed_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# seed_list=[0,1,2,3,4]
# seed_list=[0,1,2]
# seed_list=[0,1]
seed_list = [0]
print("number of seeds:  "+ str(len(seed_list)))

# data set split
flag_list = [0, 1]  # ratio or num

if data_set_name == 'IP':
    ratio_list = [Rr, 0.005]
    ratio = Rr*100
elif data_set_name == 'UP':
    ratio_list = [Rr, 0.001]
    ratio = Rr*100
elif data_set_name == 'SL':
    ratio_list = [Rr, 0.001]
    ratio = Rr*100
RRatio=ratio
num_list = [50, 0]  # [train_num,val_num]
print('Traning percentage: '+str(Rr)+'  Dataset:'+data_set_name)

windowSize = patch_size


data, gt = load_data(data_set_name, data_set_path)
data = standardization(data)
print('shape after standardization:')
print(data.shape)
data1=data


gt_reshape = gt.reshape(-1)
class_count = max(np.unique(gt))

OA_ALL = []
AA_ALL = []
KPP_ALL = []
EACH_ACC_ALL = []
Train_Time_ALL = []
Test_Time_ALL = []
CLASS_ACC = np.zeros([len(seed_list), class_count])
N_bands=np.zeros(len(seed_list))
T_NER=np.zeros(len(seed_list))
T_transform_NER_train=np.zeros(len(seed_list));T_train_ROCKET_NER=np.zeros(len(seed_list))
T_transform_NER_test=np.zeros(len(seed_list));T_test_ROCKET_NER=np.zeros(len(seed_list))
P_NER=np.zeros(len(seed_list));T_Training_NER=np.zeros(len(seed_list))  ;T_Test_NER=np.zeros(len(seed_list))

METRICS_NER=('empty',);ECA_all_NER=('empty',);METRICS_AMS=('empty',);ECA_all_AMS=('empty',);TIME_AMS=('empty',);TIME_NER=('empty',)
Alpha=[]
for curr_seed in seed_list:
    print('Traning percentage: '+str(Rr)+'  Dataset:'+data_set_name+'  Num_Seed: '+str(curr_seed+1))

    train_data_index, test_data_index, all_data_index = sampling(ratio_list,
                                                                                    num_list,
                                                                                    gt_reshape,
                                                                                    class_count,
                                                                                    flag_list[0])
    print("patch size :  "+str(windowSize))
    print("========S_NER ROCKET===========")
    print(len(train_data_index))

    if DR_flag==0:
      DIM_REDUCE="NER"+str(cc)
      print("Dimension Reduction method is: "+DIM_REDUCE)
      start=time.time()

      SET_NER = S_NER(train_data_index, data1, gt,cc)
      end=time.time()

      N_bands[curr_seed]=len(SET_NER)
      T_NER[curr_seed]=end-start

      print('S-NER finished:\n N_bands %d, Time %.2f s'
            % (N_bands[curr_seed], T_NER[curr_seed]))
      print('shape before NER BS:')
      print(data1.shape)
      print('shape after NER BS:')
      print(data1[:,:,SET_NER].shape)


      X_train_NER, y_train_NER = createPatches_concatenated(data1[:,:,SET_NER], gt, train_data_index, windowSize=windowSize, removeZeroLabels = False)
      X_train_NER=X_train_NER.astype(np.float32);y_train_NER=y_train_NER.astype(int)


      print('shape train:')
      print(X_train_NER.shape)

      X_test_NER, y_test_NER = createPatches_concatenated(data1[:,:,SET_NER], gt, test_data_index, windowSize=windowSize, removeZeroLabels = False)
      X_test_NER=X_test_NER.astype(np.float32);y_test_NER = y_test_NER.astype(int)

      print('shape test:')
      print(X_test_NER.shape)

    if DR_flag==5:
      DIM_REDUCE="NEET"
      print("Dimension Reduction method is: "+DIM_REDUCE)
      print('data shape :')
      print(data1.shape)


      X_train_NER, y_train_NER = createPatches_concatenated(data1[:,:,:], gt, train_data_index, windowSize=windowSize, removeZeroLabels = False)
      X_train_NER=X_train_NER.astype(np.float32);y_train_NER=y_train_NER.astype(int)


      print('shape train:')
      print(X_train_NER.shape)

      X_test_NER, y_test_NER = createPatches_concatenated(data1[:,:,:], gt, test_data_index, windowSize=windowSize, removeZeroLabels = False)
      X_test_NER=X_test_NER.astype(np.float32);y_test_NER = y_test_NER.astype(int)

      print('shape test:')
      print(X_test_NER.shape)


    X_train_NER = np.reshape(X_train_NER, (X_train_NER.shape[0],1,X_train_NER.shape[1]))

    mrf = MiniRocket(num_kernels=N_kernels,n_jobs=-1)

    start=time.time()
    mrf.fit(X_train_NER)
    X_train_transform_NER = mrf.transform(X_train_NER)


    print('shape train transform:')
    print(X_train_transform_NER.shape)

    scaler = StandardScaler()
    scaler.fit(X_train_transform_NER)
    X_train_transform_NER=scaler.transform(X_train_transform_NER)

    end=time.time()
    T_transform_NER_train[curr_seed] =end-start
    del X_train_NER
    start=time.time()
    classifier_NER = RidgeClassifierCV(alphas = np.logspace(-3, 3, 10))
    classifier_NER.fit(X_train_transform_NER, y_train_NER)


    end=time.time()
    T_train_ROCKET_NER[curr_seed]=end-start
    print('training finished:\n time transform train data %.2f, time train Ridge %.2f s'
          % (T_transform_NER_train[curr_seed], T_train_ROCKET_NER[curr_seed]))

    X_test_NER = np.reshape(X_test_NER, (X_test_NER.shape[0],1,X_test_NER.shape[1]))
    start=time.time()
    X_test_transform_NER = mrf.transform(X_test_NER)
    print('shape test transform:')
    print(X_test_transform_NER.shape)

    # scalert = StandardScaler()
    # scalert.fit(X_test_transform_NER)
    X_test_transform_NER=scaler.transform(X_test_transform_NER)

    end=time.time()

    T_transform_NER_test[curr_seed]+=end-start


    start=time.time()
    predictions_NER = classifier_NER.predict(X_test_transform_NER)
    end=time.time()
    T_test_ROCKET_NER[curr_seed]+=end-start

    print('Test finished: T_transform_test %.2f, T_test_ROCKET %.2f s'
          % (T_transform_NER_test[curr_seed], T_test_ROCKET_NER[curr_seed]))

    #####ACCURACY
    P_NER[curr_seed] = (predictions_NER == y_test_NER).mean()*100
    print("ROCKET"+ " = " + "{:.2f}".format(P_NER[curr_seed])) # or classifier.score(X_test_transform, Y_test)

    print('==============NER-ROCKET Done=======')

    del X_test_transform_NER,  classifier_NER, X_train_transform_NER, mrf
    del X_test_NER, y_train_NER

    T_Training_NER[curr_seed]=T_NER[curr_seed]+T_transform_NER_train[curr_seed]+T_train_ROCKET_NER[curr_seed]
    T_Test_NER[curr_seed]=T_transform_NER_test[curr_seed]+T_test_ROCKET_NER[curr_seed]

    print('TIMES:\n  T_test %.2f, T_Train %.2f s'
          % ( T_Test_NER[curr_seed], T_Training_NER[curr_seed]))

    TIME_NER=(T_NER,T_transform_NER_train,T_train_ROCKET_NER,T_transform_NER_test,T_test_ROCKET_NER,T_Training_NER,T_Test_NER)
    OA_NER = metrics.accuracy_score(y_test_NER, predictions_NER)
    confusion_matrix_NER = metrics.confusion_matrix(predictions_NER, y_test_NER)
    ECA_NER, AA_NER = AA_ECA(confusion_matrix_NER)
    kappa_NER = metrics.cohen_kappa_score(predictions_NER, y_test_NER)


    METRICS_NER = METRICS_NER + ([OA_NER,AA_NER,kappa_NER],); ECA_all_NER = ECA_all_NER + (ECA_NER,)
    # tic1 = time.perf_counter()

# resultant_data_NER = (METRICS_NER,ECA_all_NER,N_bands,TIME_NER,Alpha)
# import pickle
# # file_name = 'NER_HSI_newGPUBS_' + data_set_name  +\
# #          '_TR_prc_' + str(RRatio)  + "_DRmethod_"+DIM_REDUCE+'.pkl'
# file_name = 'GPU_FAST_regular_NER_HSI_orginal_' + data_set_name  +\
#          '_TR_prc_' + str(RRatio)  + "_DRmethod_"+DIM_REDUCE+ "_Coef_"+str(cc)+'.pkl'
# file_address = "/content/gdrive/My Drive/Colab Notebooks/HSI/Result/Finals_10_iter/"
# with open(file_address + file_name, 'wb') as f:  # Python 3: open(..., 'wb')
#   pickle.dump(resultant_data_NER, f)
# # resultant_data = (METRICS_AMS,ECA_all_AMS,TIME_AMS)

# file_name = 'AMS_HSI_newGPUBS' + data_set_name  +\
#          '_TR_prc_' + str(RRatio)  + "_DRmethod_"+DIM_REDUCE+ '.pkl'
# file_address = "/content/gdrive/My Drive/Colab Notebooks/HSI/Result/"
# with open(file_address + file_name, 'wb') as f:  # Python 3: open(..., 'wb')
#   pickle.dump(resultant_data, f)

IFFF DRs in NER and AMS using GPU:
cuda:0
number of seeds:  1
Traning percentage: 0.01  Dataset:IP
shape after standardization:
(145, 145, 200)
Traning percentage: 0.01  Dataset:IP  Num_Seed: 1
patch size :  9
========S_NER ROCKET===========
108
Dimension Reduction method is: NER0.01


<ipython-input-4-8809bb4aa12e>:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  AA[k-1,k-1]=dcAb


S-NER Done
S-NER finished:
 N_bands 62, Time 0.36 s
shape before NER BS:
(145, 145, 200)
shape after NER BS:
(145, 145, 62)
shape train:
(108, 5022)
shape test:
(10141, 5022)
shape train transform:
(108, 9996)
training finished:
 time transform train data 1.83, time train Ridge 0.11 s
shape test transform:
(10141, 9996)
Test finished: T_transform_test 119.93, T_test_ROCKET 0.69 s
ROCKET = 78.53
==============NER-ROCKET Done=======
TIMES:
  T_test 120.62, T_Train 2.31 s
